# `Dynamic AI Knowledge Assistant`

In [24]:
# ============================================================
# Phase 5 – Embedding Generation
# Step 1: Imports
# ============================================================

from pathlib import Path
import hashlib
import time
import json

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [25]:
# ============================================================
# Step 2: Project Paths
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant")
DATA_DIR = PROJECT_ROOT / "data"
METADATA_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"
SEMANTIC_DIR = (
    PROCESSED_DIR
    / "chunks"
    / "semantic"
)
CHUNK_METADATA_DIR = (
    PROCESSED_DIR
    / "chunk_metadata"
)
EMBEDDINGS_DIR = (
    PROCESSED_DIR
    / "embeddings"
)
EMBEDDINGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Embeddings directory:")
print(EMBEDDINGS_DIR)

Embeddings directory:
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings


In [26]:
# ============================================================
# Step 3: Load Phase 3 Metadata
# ============================================================

METADATA_PATH = (METADATA_DIR / "metadata.csv")
metadata_df = pd.read_csv(METADATA_PATH)
print("Metadata shape:", metadata_df.shape)
display(metadata_df.head())

Metadata shape: (128, 6)


,Article Name,URL,Category,Word Count,Character Count,Download Date
0,Artificial intelligence,https://en.wikipedia.org/wiki/Artificial_intel...,Artificial Intelligence,13223,87174,2026-08-12 21:03:09.965773
1,Artificial general intelligence,https://en.wikipedia.org/wiki/Artificial_gener...,Artificial Intelligence,5602,35819,2026-08-12 21:03:10.988638
2,A.I. Artificial Intelligence,https://en.wikipedia.org/wiki/A.I._Artificial_...,Artificial Intelligence,4226,26063,2026-08-12 21:03:12.008079
3,Timeline of artificial intelligence,https://en.wikipedia.org/wiki/Timeline_of_arti...,Artificial Intelligence,981,7231,2026-08-12 21:03:13.038940
4,History of artificial intelligence,https://en.wikipedia.org/wiki/History_of_artif...,Artificial Intelligence,14430,95762,2026-08-12 21:03:14.057315


In [27]:
# ============================================================
# Step 4: Load Existing Phase 4 Semantic Files
# ============================================================

semantic_files = sorted(
    p for p in SEMANTIC_DIR.glob("*_semantic.txt")
    if p.is_file()
)
print("Semantic files:", len(semantic_files))
assert len(semantic_files) == 128


Semantic files: 128


In [28]:
# ============================================================
# Step 5: Build Chunk Metadata
# ============================================================

def generate_article_id(file_name):
    digest = hashlib.sha256(
        file_name.encode("utf-8")
    ).hexdigest()[:12]

    return f"ART_{digest}"


metadata_lookup = {}
for _, row in metadata_df.iterrows():
    original_file_name = (
        str(row["Article Name"])
        .replace(" ", "_")
        + ".txt"
    )

    metadata_lookup[original_file_name] = {
        "article_id": generate_article_id(
            original_file_name
        ),
        "title": row["Article Name"],
        "category": row["Category"],
        "source_url": row["URL"],
        "original_file_name": original_file_name
    }

chunk_records = []
for semantic_path in semantic_files:
    original_file_name = semantic_path.name.replace(
        "_semantic.txt",
        ".txt"
    )
    if original_file_name not in metadata_lookup:
        raise ValueError(
            f"Missing metadata: {original_file_name}"
        )
    article = metadata_lookup[
        original_file_name
    ]
    with open(
        semantic_path,
        "r",
        encoding="utf-8"
    ) as f:
        content = f.read()

    #  This uses the already-established Phase 4 semantic chunk boundaries.

    chunks = content.split("\n\n")
    for chunk_index, chunk_text in enumerate(chunks):
        chunk_id = (
            f"{article['article_id']}"
            f"_chunk_{chunk_index:04d}"
        )
        chunk_records.append({
            "chunk_id": chunk_id,
            "article_id": article["article_id"],
            "title": article["title"],
            "category": article["category"],
            "source_url": article["source_url"],
            "original_file_name": article[
                "original_file_name"
            ],
            "chunk_index": chunk_index,
            "chunk_text": chunk_text,
            "chunk_length": len(chunk_text)
        })

phase5_df = pd.DataFrame(chunk_records)
print("Dataset shape:", phase5_df.shape)


Dataset shape: (4693, 9)


In [29]:
# ============================================================
# Step 6: Phase 5 Input Validation
# ============================================================

assert len(phase5_df) == 4693
assert phase5_df["chunk_id"].nunique() == 4693
assert phase5_df["article_id"].nunique() == 128
assert phase5_df["title"].nunique() == 128
assert phase5_df["source_url"].nunique() == 128
assert phase5_df["chunk_text"].notna().all()
assert (
    phase5_df["chunk_text"]
    .str.strip()
    .ne("")
    .all()
)
assert (
    phase5_df["chunk_length"]
    == phase5_df["chunk_text"].str.len()
).all()


In [30]:
# ============================================================
# Step 7: Save Chunk Metadata
# ============================================================

CHUNK_METADATA_PATH = (
    CHUNK_METADATA_DIR
    / "chunk_metadata.csv"
)

phase5_df.to_csv(
    CHUNK_METADATA_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved:")
print(CHUNK_METADATA_PATH)

print("Rows:", len(phase5_df))

Saved:
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\chunk_metadata\chunk_metadata.csv
Rows: 4693


In [31]:
# ============================================================
# Step 8: Embedding Model 1
# ============================================================

MODEL_1_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model_1 = SentenceTransformer(
    MODEL_1_NAME
)

print("Model 1 loaded:")
print(MODEL_1_NAME)

print(
    "Embedding dimension:",
    model_1.get_sentence_embedding_dimension()
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3437.49it/s]


Model 1 loaded:
sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\RAaM\AppData\Local\Temp\ipykernel_21628\817130991.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model_1.get_sentence_embedding_dimension()


In [32]:

# ============================================================
# Generate embeddings:
# ============================================================

texts = phase5_df["chunk_text"].tolist()

start_time = time.perf_counter()

embeddings_1 = model_1.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_time_1 = (
    time.perf_counter() - start_time
)

print("Embedding shape:", embeddings_1.shape)
print(
    "Embedding generation time:",
    round(embedding_time_1, 2),
    "seconds"
)

Batches: 100%|██████████| 147/147 [01:57<00:00,  1.25it/s]

Embedding shape: (4693, 384)
Embedding generation time: 117.43 seconds


In [33]:
# ============================================================
# Save Model 1 Embeddings
# ============================================================

MODEL_1_OUTPUT = (
    EMBEDDINGS_DIR
    / "all_MiniLM_L6_v2_embeddings.npy"
)

np.save(
    MODEL_1_OUTPUT,
    embeddings_1
)

print("Saved:", MODEL_1_OUTPUT)

Saved: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings\all_MiniLM_L6_v2_embeddings.npy


In [34]:
# ============================================================
# Step 10: Embedding Model 2
# ============================================================

MODEL_2_NAME = "sentence-transformers/all-mpnet-base-v2"

model_2 = SentenceTransformer(
    MODEL_2_NAME
)

print("Model 2 loaded:")
print(MODEL_2_NAME)

print(
    "Embedding dimension:",
    model_2.get_sentence_embedding_dimension()
)

start_time = time.perf_counter()

embeddings_2 = model_2.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_time_2 = (
    time.perf_counter() - start_time
)

print("Embedding shape:", embeddings_2.shape)

print(
    "Embedding generation time:",
    round(embedding_time_2, 2),
    "seconds"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11717.91it/s]
C:\Users\RAaM\AppData\Local\Temp\ipykernel_21628\3393811528.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model_2.get_sentence_embedding_dimension()


Model 2 loaded:
sentence-transformers/all-mpnet-base-v2
Embedding dimension: 768


Batches: 100%|██████████| 147/147 [15:32<00:00,  6.34s/it]

Embedding shape: (4693, 768)
Embedding generation time: 932.49 seconds


In [35]:
# ============================================================
# Save Model 2 Embeddings
# ============================================================

MODEL_2_OUTPUT = (
    EMBEDDINGS_DIR
    / "all_mpnet_base_v2_embeddings.npy"
)

np.save(
    MODEL_2_OUTPUT,
    embeddings_2
)

print("Saved:", MODEL_2_OUTPUT)

Saved: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings\all_mpnet_base_v2_embeddings.npy


In [36]:
# ============================================================
# Step 12: Embedding Model Comparison
# ============================================================

embedding_comparison = pd.DataFrame([
    {
        "Model": MODEL_1_NAME,
        "Dimensions": embeddings_1.shape[1],
        "Chunks": embeddings_1.shape[0],
        "Generation_Time_Seconds": round(
            embedding_time_1,
            2
        ),
        "Embeddings_Per_Second": round(
            len(texts) / embedding_time_1,
            2
        ),
        "Normalized": True
    },
    {
        "Model": MODEL_2_NAME,
        "Dimensions": embeddings_2.shape[1],
        "Chunks": embeddings_2.shape[0],
        "Generation_Time_Seconds": round(
            embedding_time_2,
            2
        ),
        "Embeddings_Per_Second": round(
            len(texts) / embedding_time_2,
            2
        ),
        "Normalized": True
    }
])

display(embedding_comparison)

,Model,Dimensions,Chunks,Generation_Time_Seconds,Embeddings_Per_Second,Normalized
0,sentence-transformers/all-MiniLM-L6-v2,384,4693,117.43,39.96,True
1,sentence-transformers/all-mpnet-base-v2,768,4693,932.49,5.03,True


In [39]:
# ============================================================
# Final Embedding Validation
# ============================================================

print("=" * 60)
print("PHASE 5 – EMBEDDING VALIDATION")
print("=" * 60)

# ------------------------------------------------------------
# Chunk metadata
# ------------------------------------------------------------

assert len(phase5_df) == 4693
assert phase5_df["chunk_id"].nunique() == 4693
assert phase5_df["article_id"].nunique() == 128

assert phase5_df["chunk_text"].notna().all()
assert phase5_df["chunk_text"].str.strip().ne("").all()

assert (
    phase5_df["chunk_length"]
    == phase5_df["chunk_text"].str.len()
).all()

print("Chunk metadata validation : PASSED")

# ------------------------------------------------------------
# Model 1
# ------------------------------------------------------------

assert embeddings_1.shape == (4693, 384)
assert np.isfinite(embeddings_1).all()

print("MiniLM embedding shape    : PASSED")
print("MiniLM NaN/Inf validation  : PASSED")

# ------------------------------------------------------------
# Model 2
# ------------------------------------------------------------

assert embeddings_2.shape == (4693, 768)
assert np.isfinite(embeddings_2).all()

print("MPNet embedding shape      : PASSED")
print("MPNet NaN/Inf validation   : PASSED")

print("\nAll Phase 5 embedding validations: PASSED")

PHASE 5 – EMBEDDING VALIDATION
Chunk metadata validation : PASSED
MiniLM embedding shape    : PASSED
MiniLM NaN/Inf validation  : PASSED
MPNet embedding shape      : PASSED
MPNet NaN/Inf validation   : PASSED

All Phase 5 embedding validations: PASSED


In [ ]:
# ------------------------------------------------------------
# Save the benchmark
# ------------------------------------------------------------

BENCHMARK_PATH = (
    EMBEDDINGS_DIR
    / "embedding_model_comparison.csv"
)

embedding_comparison.to_csv(
    BENCHMARK_PATH,
    index=False
)

print("Saved:")
print(BENCHMARK_PATH)

Saved benchmark:
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings\embedding_model_comparison.csv


In [41]:
phase5_summary = {
    "total_articles": 128,
    "total_chunks": 4693,
    "chunk_metadata_rows": len(phase5_df),

    "model_1": {
        "name": MODEL_1_NAME,
        "dimensions": int(embeddings_1.shape[1]),
        "chunks": int(embeddings_1.shape[0]),
        "generation_time_seconds": round(
            embedding_time_1, 2
        ),
        "embeddings_per_second": round(
            len(texts) / embedding_time_1, 2
        )
    },

    "model_2": {
        "name": MODEL_2_NAME,
        "dimensions": int(embeddings_2.shape[1]),
        "chunks": int(embeddings_2.shape[0]),
        "generation_time_seconds": round(
            embedding_time_2, 2
        ),
        "embeddings_per_second": round(
            len(texts) / embedding_time_2, 2
        )
    },

    "validation_status": "PASSED"
}

print(json.dumps(
    phase5_summary,
    indent=4
))

{
    "total_articles": 128,
    "total_chunks": 4693,
    "chunk_metadata_rows": 4693,
    "model_1": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "dimensions": 384,
        "chunks": 4693,
        "generation_time_seconds": 117.43,
        "embeddings_per_second": 39.96
    },
    "model_2": {
        "name": "sentence-transformers/all-mpnet-base-v2",
        "dimensions": 768,
        "chunks": 4693,
        "generation_time_seconds": 932.49,
        "embeddings_per_second": 5.03
    },
    "validation_status": "PASSED"
}
